<a href="https://colab.research.google.com/github/smsag99/Thesis/blob/main/codes/FeatureEng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load Animal Data


Load the dataset from '/content/drive/MyDrive/Thesis_Data/Dati_ok.txt' into a pandas DataFrame.


In [3]:
datasetAnimalPath = '/content/drive/MyDrive/Thesis_Data/Dati_ok.txt'

In [22]:
import pandas as pd

# 1. Define your column names
col_names = [
    'Farm_Code', 'Animal_ID', 'dtb', 'dtc', 'dtt',
    'parity', 'milk_kg', 'fat_p', 'protein_p', 'cells', 'NM'
]

cleaned_rows = []

# 2. Read and clean the file line by line
with open(datasetAnimalPath, 'r') as file:
    # Skip the header line
    next(file)

    for line in file:
        # Remove whitespace/newlines from ends
        line = line.strip()

        # Skip empty lines if any exist
        if not line:
            continue

        # Split the line by comma
        parts = line.split(',')

        # LOGIC: Fix the rows based on length
        if len(parts) == 11:
            # This is a correct row, keep it as is
            cleaned_rows.append(parts)

        elif len(parts) == 12:
            # This is a broken row.
            # This leaves one empty slot for 'dtb' and shifts the rest back.
            parts.pop(2)
            cleaned_rows.append(parts)

# 3. Create the DataFrame
df_animal = pd.DataFrame(cleaned_rows, columns=col_names)

# 4. Convert columns to correct types (since they were read as strings)
numeric_cols = ['Farm_Code','parity', 'milk_kg', 'fat_p', 'protein_p', 'cells', 'NM']
for col in numeric_cols:
    df_animal[col] = pd.to_numeric(df_animal[col], errors='coerce')

# 5. Convert Date Columns
date_cols = ['dtb', 'dtc', 'dtt']
for col in date_cols:
    df_animal[col] = pd.to_datetime(df_animal[col], format='%Y%m%d', errors='coerce')

# Check the result
print(df_animal.head(5))
print(f"\nTotal rows loaded: {len(df_animal)}")

   Farm_Code       Animal_ID        dtb        dtc        dtt  parity  \
0     521513  IT003990094624 2013-10-29 2022-06-07 2022-06-20       6   
1     521513  IT003990094624 2013-10-29 2022-06-07 2022-07-19       6   
2     521513  IT003990094624 2013-10-29 2022-06-07 2022-09-12       6   
3     521513  IT003990094624 2013-10-29 2022-06-07 2022-10-13       6   
4     521513  IT003990094624 2013-10-29 2022-06-07 2022-11-15       6   

   milk_kg  fat_p  protein_p  cells  NM  
0      141    617        484    656   2  
1      168    682        417    827   2  
2      134    763        459   1144   2  
3      108    663        454   1403   2  
4       94    688        451   1805   2  

Total rows loaded: 2593267


### Scaling the Numeric Data

In [23]:
df_animal['milk_kg'] = df_animal['milk_kg'] /10
df_animal['fat_p'] = df_animal['fat_p'] / 100
df_animal['protein_p'] = df_animal['protein_p'] / 100
# df_animal['cells'] = df_animal['cells'] * 100000

In [28]:
df_animal[df_animal['cells']==0] = 1

/tmp/ipython-input-2044939430.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df_animal[df_animal['cells']==0] = 1
/tmp/ipython-input-2044939430.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df_animal[df_animal['cells']==0] = 1
/tmp/ipython-input-2044939430.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df_animal[df_animal['cells']==0] = 1


### Adding ECM, AFC, SCS, DIM

In [33]:
import numpy as np
df_animal['ECM'] = (((((df_animal['fat_p'] * 10) - 40) + ((df_animal['protein_p'] * 10) - 31)) * 0.01155) + 1) * df_animal['milk_kg']
df_animal['SCS'] = np.log2((df_animal['cells']) / 100) + 3
df_animal['AFC'] = df_animal['dtc'] - df_animal['dtb']
df_animal['DIM'] = df_animal['dtt'] - df_animal['dtc']

## Loading Farm data
